# 🌉 Bridge Damage Detection — YOLO Notebook (Final)
**Model:** YOLOv8-seg & YOLOv11-seg | **Dataset:** dacl10k | **Classes:** 19

| Step | Deskripsi |
|---|---|
| 1 | Cek GPU + Mount Google Drive |
| 2 | Install Ultralytics |
| 3 | Setup + Import helper |
| 4 | Konversi dataset → YOLO format (sekali saja) |
| 5 | Buat konfigurasi YAML |
| 6 | Training YOLOv8-seg (auto-resume) |
| 7 | Training YOLOv11-seg (auto-resume) |
| 8 | Evaluasi seragam (mIoU, F1, FPS) |
| 9 | Robustness evaluation |
| 10 | Visualisasi prediksi |
| 11 | Gabungkan semua hasil benchmark |

> **Fix yang sudah diintegrasikan:**
> - Auto-resume dari `last.pt` otomatis
> - `extract_logits()` helper — handle berbagai output format
> - Evaluasi menggunakan `JaccardIndex(task='multiclass')` seragam
> - FPS measurement via loop 50 iterasi
> - Konversi dataset otomatis skip jika sudah ada

## ✅ Step 1 — Cek GPU & Mount Google Drive

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys

PROJECT_PATH = '/content/drive/MyDrive/Penelitian/bridge-benchmark-cv'
if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
    print(f'Working directory: {os.getcwd()}')
else:
    print(f'ERROR: {PROJECT_PATH} tidak ditemukan!')

## 📦 Step 2 — Install Ultralytics

In [ ]:
!pip install -q ultralytics
!pip install -q wandb
!pip install -q "torchmetrics>=1.3.0" pycocotools albumentations

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics {ultralytics.__version__} OK')

## 🔧 Step 3 — Setup + Import Helper

In [ ]:
import os, sys, importlib.util, torch
import torch.nn.functional as F

for folder in ['train','datasets','models','eval']:
    init = os.path.join(folder, '__init__.py')
    os.makedirs(folder, exist_ok=True)
    if not os.path.exists(init):
        open(init,'w').write('')

def import_from_file(module_name, file_path):
    if module_name in sys.modules: del sys.modules[module_name]
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

def extract_logits(outputs, target_size):
    """Ekstrak & upsample logits dari berbagai format output model."""
    if hasattr(outputs, 'logits'):
        logits = outputs.logits
    elif isinstance(outputs, (list, tuple)):
        logits = outputs[0]
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
    elif isinstance(outputs, dict):
        logits = outputs.get('logits', outputs.get('out'))
    else:
        logits = outputs
    if logits.ndim == 3:
        logits = logits.unsqueeze(0)
    if logits.shape[-2:] != target_size:
        logits = F.interpolate(logits, size=target_size,
                               mode='bilinear', align_corners=False)
    return logits

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Setup OK!')

## 🔑 Step 3b — WandB Login

> API key dibaca otomatis dari `wandb-key.cfg`.
> File ini di-gitignore — tidak ter-upload ke GitHub.

In [ ]:
import wandb
from pathlib import Path

_cfg = Path('wandb-key.cfg')
if _cfg.exists():
    _lines = _cfg.read_text().strip().splitlines()
    _api_key = _lines[1].strip()
    wandb.login(key=_api_key)
    print('WandB login OK')
    print('Project : bridge-benchmark-cv')
else:
    print('WARNING: wandb-key.cfg tidak ditemukan!')
    wandb.login()


In [ ]:
loader_mod = import_from_file('datasets.dacl10k_loader', 'datasets/dacl10k_loader.py')
build_dataloaders   = loader_mod.build_dataloaders
build_robust_loader = loader_mod.build_robust_loader
ALL_CLASSES = loader_mod.ALL_CLASSES
NUM_CLASSES = loader_mod.NUM_CLASSES

IMG_SIZE    = 512
BATCH_SIZE  = 8
DATA_ROOT   = 'data/dacl10k'

_, _, test_loader = build_dataloaders(
    root_dir=DATA_ROOT, img_size=IMG_SIZE,
    batch_size=BATCH_SIZE, num_workers=2
)
print(f'Test loader: {len(test_loader)} batches | {NUM_CLASSES} kelas')

## 🔄 Step 4 — Konversi Dataset → YOLO Format

> Skip otomatis jika `data/dacl10k_yolo/` sudah ada.

In [ ]:
import json, shutil
from pathlib import Path
from tqdm.notebook import tqdm

YOLO_ROOT = Path('data/dacl10k_yolo')
CLASS_TO_IDX = {name:idx for idx,name in enumerate(ALL_CLASSES)}

# Cek apakah sudah dikonversi
if (YOLO_ROOT/'images'/'train').exists():
    n = len(list((YOLO_ROOT/'images'/'train').glob('*.*')))
    print(f'Dataset YOLO sudah ada ({n} gambar di train/) — skip konversi.')
else:
    print('Mulai konversi Supervisely → YOLO format...')

    def convert_split(split_src, split_dst):
        img_dir = Path(DATA_ROOT)/split_src/'img'
        ann_dir = Path(DATA_ROOT)/split_src/'ann'
        out_img = YOLO_ROOT/'images'/split_dst
        out_lbl = YOLO_ROOT/'labels'/split_dst
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)

        img_files = sorted(img_dir.glob('*.*'))
        for img_path in tqdm(img_files, desc=f'[{split_src}]'):
            dst = out_img/img_path.name
            if not dst.exists(): shutil.copy2(img_path, dst)

            ann_path = ann_dir/(img_path.name+'.json')
            if not ann_path.exists(): continue

            with open(ann_path) as f: ann = json.load(f)
            h,w = ann['size']['height'], ann['size']['width']
            lines = []
            for obj in ann.get('objects',[]):
                cls = obj.get('classTitle','')
                if cls not in CLASS_TO_IDX: continue
                ext = obj.get('points',{}).get('exterior',[])
                if len(ext) < 3: continue
                pts = []
                for px,py in ext:
                    pts.append(round(max(0,min(1,px/w)),6))
                    pts.append(round(max(0,min(1,py/h)),6))
                lines.append(str(CLASS_TO_IDX[cls])+' '+' '.join(map(str,pts)))
            (out_lbl/(img_path.stem+'.txt')).write_text('\n'.join(lines))
        return len(img_files)

    n_tr = convert_split('train','train')
    n_v  = convert_split('val','val')
    n_te = convert_split('test','test')
    print(f'\nKonversi selesai: train={n_tr} | val={n_v} | test={n_te}')

## 📄 Step 5 — Konfigurasi YAML

In [ ]:
import yaml
from pathlib import Path

YOLO_ROOT = Path('data/dacl10k_yolo')
yaml_path = YOLO_ROOT/'dacl10k.yaml'

cfg = {
    'path'  : str(YOLO_ROOT.resolve()),
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : NUM_CLASSES,
    'names' : ALL_CLASSES,
}
with open(yaml_path,'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print(f'YAML OK: {yaml_path}')
print(f'nc={NUM_CLASSES} kelas')

## 🚀 Step 6 — Training YOLOv8-seg (Auto-Resume)

> Auto-resume dari `last.pt` jika session terputus.

In [ ]:
from ultralytics import YOLO
import os, torch

os.makedirs('outputs/yolov8', exist_ok=True)
LAST_V8 = 'outputs/yolov8/weights/last.pt'
BEST_V8 = 'outputs/yolov8/weights/best.pt'

TRAIN_CFG = dict(
    data=str(yaml_path), epochs=50, imgsz=512,
    batch=BATCH_SIZE, optimizer='AdamW',
    lr0=1e-4, lrf=0.01, warmup_epochs=3, patience=10,
    project='outputs', name='yolov8', exist_ok=True,
    seed=42, amp=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2, verbose=True,
)

if os.path.exists(LAST_V8):
    print(f'Checkpoint ditemukan — resume YOLOv8...')
    model_v8 = YOLO(LAST_V8)
    results_v8 = model_v8.train(resume=True)
else:
    print('Training YOLOv8 dari awal...')
    model_v8 = YOLO('yolov8m-seg.pt')
    results_v8 = model_v8.train(**TRAIN_CFG)

print(f'\nTraining YOLOv8 selesai! Best: {BEST_V8}')

## 🚀 Step 7 — Training YOLOv11-seg (Auto-Resume)

In [ ]:
os.makedirs('outputs/yolov11', exist_ok=True)
LAST_V11 = 'outputs/yolov11/weights/last.pt'
BEST_V11 = 'outputs/yolov11/weights/best.pt'

TRAIN_CFG_11 = {**TRAIN_CFG, 'name':'yolov11'}

if os.path.exists(LAST_V11):
    print(f'Checkpoint ditemukan — resume YOLOv11...')
    model_v11 = YOLO(LAST_V11)
    results_v11 = model_v11.train(resume=True)
else:
    print('Training YOLOv11 dari awal...')
    model_v11 = YOLO('yolo11m-seg.pt')
    results_v11 = model_v11.train(**TRAIN_CFG_11)

print(f'\nTraining YOLOv11 selesai! Best: {BEST_V11}')

## 📊 Step 8 — Evaluasi Seragam (mIoU, F1, FPS, Params)

> Menggunakan metrik identik dengan notebook PyTorch.

In [ ]:
import time, pandas as pd
from torchmetrics import JaccardIndex
from torchmetrics.classification import MulticlassF1Score
from tqdm.notebook import tqdm
from ultralytics import YOLO

def evaluate_yolo(model_name, weights_path):
    print(f'\nEvaluasi {model_name} ({weights_path})...')

    yolo = YOLO(weights_path)
    yolo.model = yolo.model.to(DEVICE)
    yolo.model.eval()
    n_params = sum(p.numel() for p in yolo.model.parameters())/1e6

    m_iou = JaccardIndex(task='multiclass', num_classes=NUM_CLASSES,
                         average='none', ignore_index=255).to(DEVICE)
    m_f1  = MulticlassF1Score(num_classes=NUM_CLASSES,
                              average='none', ignore_index=255).to(DEVICE)

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'  {model_name}'):
            images  = batch['image'].to(DEVICE)
            targets = batch['mask'].to(DEVICE)
            outputs = yolo.model(images)
            logits  = extract_logits(outputs, targets.shape[-2:])
            preds   = logits.argmax(1)
            m_iou.update(preds, targets)
            m_f1.update(preds, targets)

    miou_pc = m_iou.compute().cpu()
    f1_pc   = m_f1.compute().cpu()
    miou    = miou_pc.mean().item()
    f1      = f1_pc.mean().item()

    # FPS
    dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        for _ in range(10): yolo.model(dummy)
        t0 = time.time()
        for _ in range(50): yolo.model(dummy)
        fps = 50/(time.time()-t0)

    print(f'  mIoU={miou:.4f} | F1={f1:.4f} | FPS={fps:.1f} | Params={n_params:.1f}M')

    # Simpan CSV
    os.makedirs('outputs/results', exist_ok=True)
    pd.DataFrame({
        'class'    : ALL_CLASSES+['MEAN'],
        'iou'      : miou_pc.tolist()+[miou],
        'f1'       : f1_pc.tolist()+[f1],
        'model'    : model_name,
        'fps'      : fps,
        'params_m' : n_params,
    }).to_csv(f'outputs/results/{model_name}_test_results.csv', index=False)

    return {'model':model_name,'miou':miou,'f1':f1,'fps':fps,
            'params_m':n_params,'miou_pc':miou_pc,'f1_pc':f1_pc}

v8_res  = evaluate_yolo('yolov8',  'outputs/yolov8/weights/best.pt')
v11_res = evaluate_yolo('yolov11', 'outputs/yolov11/weights/best.pt')

print('\n=== Ringkasan YOLO ===')
for r in [v8_res, v11_res]:
    print(f'  {r["model"]:<10} mIoU={r["miou"]:.4f} | F1={r["f1"]:.4f} | FPS={r["fps"]:.1f} | Params={r["params_m"]:.1f}M')

In [ ]:
# Per kelas
print(f'{"Class":<32} {"YOLOv8":>8} {"YOLOv11":>8} {"Delta":>8}')
print('-'*62)
for i, cls in enumerate(ALL_CLASSES):
    v8  = v8_res['miou_pc'][i].item()
    v11 = v11_res['miou_pc'][i].item()
    d   = v11-v8
    tag = ' <<' if abs(d)>0.05 else ''
    print(f'{cls:<32} {v8:>8.4f} {v11:>8.4f} {d:>+8.4f}{tag}')
print('-'*62)
print(f'{"MEAN":<32} {v8_res["miou"]:>8.4f} {v11_res["miou"]:>8.4f} {v11_res["miou"]-v8_res["miou"]:>+8.4f}')

## 🌫️ Step 9 — Robustness Evaluation

In [ ]:
all_robust = {}
for model_name, weights in [('yolov8','outputs/yolov8/weights/best.pt'),
                             ('yolov11','outputs/yolov11/weights/best.pt')]:
    yolo = YOLO(weights)
    yolo.model = yolo.model.to(DEVICE)
    yolo.model.eval()
    normal_miou = v8_res['miou'] if 'v8' in model_name else v11_res['miou']
    rr = {'normal': normal_miou}

    for corruption in ['blur','noise','brightness']:
        rl = build_robust_loader(root_dir=DATA_ROOT, corruption=corruption,
                                 batch_size=BATCH_SIZE, num_workers=2)
        m = JaccardIndex(task='multiclass', num_classes=NUM_CLASSES,
                         average='none', ignore_index=255).to(DEVICE)
        with torch.no_grad():
            for batch in tqdm(rl, desc=f'[{model_name}][{corruption}]'):
                images  = batch['image'].to(DEVICE)
                targets = batch['mask'].to(DEVICE)
                logits  = extract_logits(yolo.model(images), targets.shape[-2:])
                m.update(logits.argmax(1), targets)
        rob = m.compute().mean().item()
        rr[corruption] = rob
        print(f'  [{model_name}][{corruption:<12}] {rob:.4f}  delta={normal_miou-rob:+.4f}')
    all_robust[model_name] = rr

    pd.DataFrame([{
        'model':model_name,'normal':rr['normal'],
        'blur':rr.get('blur',0),'noise':rr.get('noise',0),'brightness':rr.get('brightness',0),
        'delta_blur':rr['normal']-rr.get('blur',rr['normal']),
        'delta_noise':rr['normal']-rr.get('noise',rr['normal']),
        'delta_brightness':rr['normal']-rr.get('brightness',rr['normal']),
    }]).to_csv(f'outputs/results/{model_name}_robustness.csv', index=False)

print('\nRobustness CSV tersimpan.')

## 🖼️ Step 10 — Visualisasi Prediksi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CLASS_COLORS = [
    (255,192,203),(178,223,138),(0,255,128),(0,0,255),(139,0,0),
    (255,140,0),(169,169,169),(207,52,118),(128,0,0),(246,124,104),
    (128,0,128),(128,128,0),(212,175,162),(253,191,111),(255,0,0),
    (123,211,247),(0,0,139),(0,128,0),(0,139,139),
]
def denormalize(t):
    img = t.permute(1,2,0).numpy()
    return (img*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406])).clip(0,1)
def mask_rgb(m):
    mask = m.numpy()
    rgb  = np.zeros((*mask.shape,3), dtype=np.uint8)
    for i,c in enumerate(CLASS_COLORS): rgb[mask==i] = c
    rgb[mask==255] = (50,50,50)
    return rgb

batch = next(iter(test_loader))
imgs  = batch['image'][:4].to(DEVICE)
tgts  = batch['mask'][:4]
preds = {}

for mn, wp in [('yolov8','outputs/yolov8/weights/best.pt'),
               ('yolov11','outputs/yolov11/weights/best.pt')]:
    yolo = YOLO(wp); yolo.model = yolo.model.to(DEVICE); yolo.model.eval()
    with torch.no_grad():
        logits = extract_logits(yolo.model(imgs), tgts.shape[-2:])
        preds[mn] = logits.argmax(1).cpu()

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('YOLOv8 vs YOLOv11 vs GT', fontsize=13, fontweight='bold')
row_labels = ['Input','Ground Truth','YOLOv8-seg','YOLOv11-seg']
rows_data  = [None, tgts, preds['yolov8'], preds['yolov11']]

for col in range(4):
    axes[0,col].imshow(denormalize(batch['image'][col]))
    axes[0,col].set_title(f'Sample {col}', fontsize=8)
    for row in range(1,4):
        axes[row,col].imshow(mask_rgb(rows_data[row][col]))
    for row in range(4): axes[row,col].axis('off')
for row,label in enumerate(row_labels):
    axes[row,0].set_ylabel(label, fontsize=10, fontweight='bold')

plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/yolo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/yolo_comparison.png')

# ── WandB: log test + robustness + figure per model ─────────────────
import wandb

for _mname, _res in [('yolov8', v8_res), ('yolov11', v11_res)]:
    _rob    = all_robust.get(_mname, {})
    _normal = _rob.get('normal', _res['miou'])

    with wandb.init(
        project = 'bridge-benchmark-cv',
        name    = _mname,
        reinit  = True,
    ) as _run:
        # Table per-class IoU
        _tbl = wandb.Table(columns=["class", "IoU", "F1"])
        for _cls, _iou, _f1 in zip(ALL_CLASSES, _res['miou_pc'], _res['f1_pc']):
            _tbl.add_data(_cls, round(_iou.item(), 4), round(_f1.item(), 4))

        _log = {
            "test/mIoU"     : _res['miou'],
            "test/mF1"      : _res['f1'],
            "test/fps"      : _res['fps'],
            "test/params_M" : _res['params_m'],
            "test/per_class": _tbl,
        }
        # Per-class IoU
        for _cls, _iou in zip(ALL_CLASSES, _res['miou_pc']):
            _log[f"test/iou/{_cls}"] = _iou.item()

        # Robustness
        for _k in ['blur', 'noise', 'brightness']:
            if _k in _rob:
                _log[f"robustness/{_k}"]       = _rob[_k]
                _log[f"robustness/delta_{_k}"] = _normal - _rob[_k]

        # Prediction figure (sama untuk kedua model)
        _fig = 'outputs/figures/yolo_comparison.png'
        if os.path.exists(_fig):
            _log["test/predictions_figure"] = wandb.Image(_fig)

        wandb.log(_log)
        print(f"[WandB] {_mname}: mIoU={_res['miou']:.4f} — semua metrics logged")

print("[WandB] Semua YOLO runs selesai.")


## 📋 Step 11 — Gabungkan Semua Hasil Benchmark

> Jalankan setelah semua model selesai (termasuk dari notebook PyTorch).

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

results_dir = Path('outputs/results')
csvs = sorted(results_dir.glob('*_test_results.csv'))

if len(csvs) == 0:
    print('Belum ada hasil.')
else:
    print(f'{len(csvs)} model ditemukan:')
    for c in csvs: print(f'  {c.name}')

    combined = pd.concat([pd.read_csv(c) for c in csvs])
    pivot    = combined.pivot(index='class', columns='model', values='iou')
    final    = pd.concat([pivot.drop('MEAN').sort_index(), pivot.loc[['MEAN']]])

    print('\n=== BENCHMARK COMPLETE ===')
    print(final.round(4).to_string())
    final.to_csv('outputs/results/benchmark_complete.csv')
    print('\nTersimpan: outputs/results/benchmark_complete.csv')

    # Bar chart
    means = final.loc['MEAN'].sort_values(ascending=False)
    cmap  = {'segformer':'#8b5cf6','swinunet':'#a78bfa',
             'yolov8':'#3b82f6','yolov11':'#60a5fa',
             'deeplabv3':'#10b981','unet_effnet':'#34d399','unet_resnet':'#6ee7b7'}
    colors = [cmap.get(m,'#94a3b8') for m in means.index]

    fig, ax = plt.subplots(figsize=(12,5))
    bars = ax.bar(means.index, means.values, color=colors, edgecolor='white')
    ax.axhline(0.42, color='red', ls='--', lw=1.5, label='Baseline dacl10k (0.42)')
    ax.set_ylabel('Mean IoU'); ax.set_title('Benchmark mIoU — dacl10k', fontweight='bold')
    ax.set_ylim(0, means.values.max()*1.15); ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/figures/benchmark_miou_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Status checklist semua model
print('=== STATUS CHECKLIST ===')
all_models = ['segformer','deeplabv3','unet_effnet','unet_resnet','yolov8','yolov11']
for mn in all_models:
    csv  = Path(f'outputs/results/{mn}_test_results.csv')
    ckpt = Path(f'outputs/{mn}/best_model.pth')
    ckpt_yolo = Path(f'outputs/{mn}/weights/best.pt')

    if csv.exists():
        df = pd.read_csv(csv)
        miou = df[df['class']=='MEAN']['iou'].values[0]
        print(f'  DONE   [{mn:<15}] mIoU={miou:.4f}')
    elif ckpt.exists() or ckpt_yolo.exists():
        print(f'  PARTIAL [{mn:<15}] Training done, evaluasi belum')
    else:
        print(f'  PENDING [{mn:<15}] Belum dimulai')